In [7]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import math 
%matplotlib qt5


class Class2dPoint:
    def __init__(self, X=None,Y=None, point_array = None):
        self.points = {}
        self.corrected_points = {}
        if not (X == Y == point_array is None): 
            for x,y,p in zip(X,Y,point_array):
                self.points[p] = {
                    "x": x,
                    "y": y
                }      
    def set_point(self, x,y,point):
        self.points[point] = {
            "x": x,
            "y": y
        }
    
    def set_corrected_points(self, x,y,point):
            self.corrected_points[point] = {
                "x": x,
                "y": y
            }

    def calculate_error(self, ground_truth):
        self.error = {}
        self.error_mean = 0
        for p in ground_truth.points:        
            self.error[p] = math.sqrt((ground_truth.points[p]['x'] - self.corrected_points[p]['x'])**2 + (ground_truth.points[p]['y'] - self.corrected_points[p]['y'])**2) * 100 # calculate euclidian distance and convert to cm
            self.error_mean += self.error[p]
        self.error_mean/=len(ground_truth.points)

    def define_rotation_matrix(self,radians):
        self.rotation_matrix = np.array([[np.cos(radians),-np.sin(radians)],
                            [np.sin(radians),np.cos(radians)]])


    def rotate_and_translate_points_based_on_fixed_beacons(self,fixed_beacons_pose):
        radians_between_1_and_2=math.atan2(fixed_beacons_pose.points['2']['y'] - fixed_beacons_pose.points['1']['y'],fixed_beacons_pose.points['2']['x'] - fixed_beacons_pose.points['1']['x'])
        radians_correction = -radians_between_1_and_2 # you want to turn to the opposite side for the correction.
        self.define_rotation_matrix(radians_correction)
            
        for p in fixed_beacons_pose.points:
            current_point = np.array([fixed_beacons_pose.points[p]['x'],fixed_beacons_pose.points[p]['y']]) 
            [new_x,new_y] = self.rotation_matrix.dot(current_point)
            if p =='1':
                translate_y = new_y
                translate_x = new_x
            new_y = new_y - translate_y
            new_x = new_x - translate_x
            fixed_beacons_pose.set_corrected_points(new_x,new_y,p)                               

        for p in self.points:
            current_point = np.array([self.points[p]['x'],self.points[p]['y']]) 
            [new_x,new_y] = self.rotation_matrix.dot(current_point)
            new_y = new_y - translate_y
            new_x = new_x - translate_x
            self.set_corrected_points(new_x,new_y,p)

    def plot_dot_graph(self,plot_corrected=None):
        fig, ax = plt.subplots()
        plt.grid()      
        if (plot_corrected):
            points_to_plot = self.corrected_points
        else:
            points_to_plot = self.points

        for p in points_to_plot:
            x = self.points[p]['x']
            y = self.points[p]['y']
            ax.plot(x,y, 'ro')                                       
            ax.annotate(f"{p}:({x}, {y})", xy=(x,y), xytext=(x, y)) 
        plt.show()

    def plot_error(self,message):
        fig, ax = plt.subplots()
        point_error_array = []
        for p in self.corrected_points:
            a = ax.bar(p, self.error[p])
            temp_point_error_array = [f"{self.error[p]:.2f}"]
            ax.bar_label(a, labels=temp_point_error_array)

        ax.set_title('Error between real and marvelmind program'f"({message})")
        ax.set_xlabel('point')
        ax.set_ylabel('error (cm)')
        plt.show()
                          


#### Defining functions part 1

In [8]:
def get_column_average(bags_path):
  point_array = ["1","2","3","4","5","6","7","7.1","7.2","7.3","7.4","7.5","8","8.1","8.2","8.3","10.1","10.2"]
  temp_class = Class2dPoint()
  for point in point_array:
    csv_file_path=bags_path+f"point_{point}/hedgehog_pos/hedgehog_pos.csv"
    df = pd.read_csv(csv_file_path,usecols=['x_m','y_m','z_m'])
    column_averages = df.mean()
    temp_class.set_point(np.round(column_averages['x_m'],4),np.round(column_averages['y_m'],4),point)
  return temp_class


#### Create real points for comparison

The real points positions were generated by the graph.cpp file.

In [9]:
## the cpp code which generates the graph considers the positions 1.2,2.2,3.2, and 4.2 the mobile beacon positions, and 1,2,3, and 4 the fixed beacon position.
# however in the bags, the points 1,2,3 and 4 are the positions of the mobile beacons.
# So to make the graph easier to understand, we are going to remove points 1,2,3 and 4 from the real graph, and 
# change the names of points 1.2,2.2,3.2, and 4.2 to 1,2,3, and 4, respectively.


X_real = [0, 0, 10.25, 10.25, 10.25, 10.25, 0, 0, 0, 5.096, 4.87, 4.87, 4.87, 4.87, 4.87, 4.87, 10.25, 12.25, 14.25, 27.788, -1.8, -3.6]
Y_real = [0, 0.042, 0.019, 0.046, 3.748, 3.7435, 3.749, 3.7445, 1.91, 3.152, 1.119, 7.3935, -0.406, 1.909, 3.794, 5.2935, 1.909, 1.909, 1.909, 1.909, 1.909, 1.909]
P_real = [1, 1.2, 2, 2.2, 3, 3.2, 4, 4.2, 5, 6, 7, 7.1, 7.2, 7.3, 7.4, 7.5, 8, 8.1, 8.2, 8.3, 10.1, 10.2]

#real_points = {}
real_points = Class2dPoint()
i=0
points_to_exclude = [1,2,3,4]
point_to_change_name = [1.2,2.2,3.2,4.2]
for point in P_real:
    if point not in points_to_exclude:
        if point in point_to_change_name:
            real_points.set_point(X_real[i],Y_real[i],f'{int(point)}')
        else:
            real_points.set_point(X_real[i],Y_real[i],f'{(point)}')
    i = i + 1

#### Reading data from bags

We define a list of points which match the real points list.
However, they only match after the removal of points 1.2,2.2,3.2, and 4.2 from the real points list.
Then, we read data from the bags:
- Bag1: we recorded the marvelmind program result after passing every fixed beacon position.
- Bag2: we recorded the marvelmind program result afte the fixed beacons distance were calculated by the marvelmind program.


In [10]:
point_array = ["1","2","3","4","5","6","7","7.1","7.2","7.3","7.4","7.5","8","8.1","8.2","8.3","10.1","10.2"]
# points 1,2,3 and 4, are not the fixed beacons point. They are the mobile beacons placed close.
# The comparison will be done between 1 marvelmind and 1.2 real (mobile beacon pose on map).
bags_path="/persistent/scripts/bags/correct_better_zone_marvelmind_bag_data_with_distances/"
marvel_with_distance_positions = get_column_average(bags_path)
bags_path="/persistent/scripts/bags/correct_better_zone_marvelmind_bag_data_without_distances/"
marvel_without_distance_positions = get_column_average(bags_path)

#### Defining functions part 2

Define functions which need data from bags.

In [ ]:


def plot_difference_between_real_and_program_poses(correct_x,corrected_y):
    fig, ax = plt.subplots()
    ax.grid()
    for x,y,p in zip(X_real, Y_real,point_array):
        ax.annotate(f"real {p}:({x}, {y})", xy=(x,y), xytext=(x, y))
        ax.plot(x,y, 'bx') 

    for x,y,p in zip(correct_x, corrected_y,point_array):
        ax.annotate(f'{p}:({x:.2f}, {y:.2f})', xy=(x,y), xytext=(x, y))
        ax.plot(x,y, 'ro') 

    ax.set_title('Real versus marvelmind pose')
    ax.set_xlabel('x (meters)')
    ax.set_ylabel('y (meters)')
    plt.show() 

### Comparison between marvelmind program and real points.


##### Strategy
1. Create ortogonal axes from (0,0)
2. Compare real to ortogonal axes, see if there is rotation and displacement. Ideally points 1,2 and 4, should form a close orthogonal axes.
    1. Calculate rotation between fixed beacons positions 1 and 2.
    2. Use a rotation matrix to rotate the points -correction.
    3. Translate the points to make position 1 approximately the origin (0,0)
3. Compare marvelmind program (passing distance) to ortogonal axes.
    1. Apply the same principals used by real.





In [11]:
## Point 8.3 had results way outside of acceptable ranges, and points 7.1 and 7.5 just repeated the pose of the last point.
## Probably, the mobile beacon lost its signal in poses 7.1 and 7.5. 
ignore_points = ['7.1','7.5','8.3'] # Some points were ignored because they were outliersarray_of_lists_to_be_cut = [marvel_with_distance_positions,marvel_without_distance_positions,real_points]
array_of_lists_to_be_cut = [marvel_with_distance_positions,marvel_without_distance_positions,real_points]
for point_to_be_ignored in ignore_points:
    for list_to_be_cut in array_of_lists_to_be_cut:
      list_to_be_cut.points.pop(point_to_be_ignored,None)


In [15]:
plt.close('all')
fixed_beacon_points = Class2dPoint([0.004,10.246,10.097,-0.140],[0.005,0.360,4.107,3.740],["1","2","3","4"])
marvel_with_distance_positions.rotate_and_translate_points_based_on_fixed_beacons(fixed_beacon_points)
marvel_with_distance_positions.calculate_error(real_points)
marvel_with_distance_positions.plot_error("with distance")

In [16]:
plt.close('all')
fixed_beacon_points_without_distance = Class2dPoint([0.001,10.112,9.991,-0.140],[0.001,0.270,4.004,3.630],["1","2","3","4"])
marvel_without_distance_positions.rotate_and_translate_points_based_on_fixed_beacons(fixed_beacon_points_without_distance)
marvel_without_distance_positions.calculate_error(real_points)
marvel_without_distance_positions.plot_error("with distance")

In [31]:

marvelmind_error_dictionary = {
    'without_distance': point_error_array2,
    'with_distance': point_error_array,
}
width = 0.6  # the width of the bars: can also be len(x) sequence


fig, ax = plt.subplots()
bottom = np.zeros(len(point_array))

for type_of_program, error_array in marvelmind_error_dictionary.items():
    p = ax.bar(point_array, error_array, width, label=type_of_program, bottom=bottom)
    bottom += error_array
    ax.bar_label(p, label_type='center')

ax.set_title('Error between marvelmind program passing or not distance between fixed beacons.')
ax.legend()

plt.show()

NameError: name 'point_error_array2' is not defined

In [38]:
diff_between_1_and_2 = np.subtract(np.array(point_error_array),np.array(point_error_array2))
fig, ax = plt.subplots()
for i in range(len(diff_between_1_and_2)):
    a = ax.bar(f"{point_array[i]}", diff_between_1_and_2[i])
    temp_point_error_array = [f"{diff_between_1_and_2[i]:.2f}"]
    ax.bar_label(a, labels=temp_point_error_array)

ax.set_title('Error between real and marvelmind program')
ax.set_xlabel('point')
ax.set_ylabel('error (cm)')
plt.show()



In [10]:
class Class2dPoint:
    def __init__(self, X=None,Y=None, point_array = None):
        self.points = {}
        if not (X == Y == point_array is None): 
            for x,y,p in zip(X,Y,point_array):
                self.points[p] = {
                    "x": x,
                    "y": y
                }      
        else:
            print("banana")
    def set_points(self, x,y,point):
        self.points[point] = {
            "x": x,
            "y": y
        }


point_array = ["1","2","3","4","5","6"]#"7","7.1","7.2","7.3","7.4","7.5","8","8.1","8.2","8.3","10.1","10.2"]
X = [1,2,3,4,5,6]
Y = [6,5,4,3,2,1]
marvelmind_points = Class2dPoint(X,Y,point_array)
print(marvelmind_points.points['1']['y'])

test = Class2dPoint()
test.set_points(1,15,"23")



print(len(marvelmind_points.points))





6
banana
6


transformar os dados em dicionários do tipo com cada posição sendo p
exemplo
real[8.1].x
with_distance[7.1].y
x,y,z